<a href="https://colab.research.google.com/github/OssamaSijbesma/uu-data-wrangling-assignment-1/blob/main/wc_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Assignment 1: Data Extraction & Integration](https://hqahtan.github.io/Teaching/infomdw/assignments/Assignment_1_26_27.html)

In [29]:
import sqlite3
import pandas as pd
import requests
from collections import defaultdict

# Set constants
GOOGLE_COLAB = True # When using Google Colab set to True.
DATABASE_URL = "https://github.com/OssamaSijbesma/uu-data-wrangling-assignment-1/raw/refs/heads/main/wc_database.db"

## Attach database

In [2]:
conn = None

if GOOGLE_COLAB:
    # Connect to in memory database
    conn = sqlite3.connect(":memory:")

    # Fetch database from online data source
    response = requests.get(DATABASE_URL)

    # Load serialized database
    conn.deserialize(response.content)
else:
    # Connect to local database file
    conn = sqlite3.connect("./wc_database.db")

# Create cursor
cursor = conn.cursor()

## Execute query

In [6]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables imported:", cursor.fetchall())

Tables imported: [('staff_role',), ('official_role',), ('stadium',), ('person',), ('team',), ('staff_member',), ('player',), ('match',), ('goal',), ('substitute',), ('card',), ('match_official',)]


## Read table into Pandas data frame

In [8]:
df = pd.read_sql_query("SELECT last_name, first_name, birth_date FROM PERSON", conn)
df.head()

,last_name,first_name,birth_date
0,Koeman,Ronald,1963-03-21
1,Van Dijk,Virgil,1991-07-08
2,De Jong,Frenkie,1997-05-12
3,Depay,Memphis,1994-02-13
4,Gakpo,Cody,1999-05-07


## Hash strings

In [30]:
def hash_text(text: str, table_cardinality = 7):
  return abs(hash(text)) % 7

hash_table = defaultdict(list)

for last_name in df['last_name']:
  hash_table[hash_text(last_name)].append(last_name)

print(hash_table)

defaultdict(<class 'list'>, {1: ['Koeman', 'Martinez', 'Martinez', 'Santos'], 3: ['Van Dijk', 'Taylor', 'Wilson'], 5: ['De Jong', 'Depay', 'Silva', 'Muller', 'Oliver'], 4: ['Gakpo', 'Garcia'], 2: ['Rice'], 6: ['Kane'], 0: ['Bellingham', 'Brown']})


## Collision rate

In [32]:
N = len(df['last_name'])
B = len(hash_table)

collision_rate = (N - B) / N
print(f"Total elements: {N}")
print(f"Occupied buckets: {B}")
print(f"Collision rate: {collision_rate:.2%}")

Total elements: 18
Occupied buckets: 7
Collision rate: 61.11%


## Close database

In [ ]:
conn.close()